# Stock News Fetcher

This notebook fetches top 10 stock headlines using `yfinance` with source links and publication dates.

In [1]:
import os
from datetime import datetime

import pandas as pd
import yfinance as yf

In [2]:
stock_input = "RELIANCE"
top_n = 10


def normalize_ticker(stock_query: str) -> str:
    ticker = stock_query.strip().upper()
    if not ticker:
        return ticker
    if ticker.startswith("^") or ticker.endswith(".NS") or ticker.endswith(".BO"):
        return ticker
    return f"{ticker}.NS"


ticker = normalize_ticker(stock_input)
ticker

'RELIANCE.NS'

In [3]:
def fetch_stock_news(stock_query: str, limit: int = 10):
    base_ticker = normalize_ticker(stock_query)
    candidates = [base_ticker, stock_query.strip().upper(), f"{stock_query.strip().upper()}.BO"]

    raw_news = []
    final_ticker = base_ticker

    for candidate in candidates:
        if not candidate:
            continue
        try:
            items = yf.Ticker(candidate).news or []
            if items:
                raw_news = items
                final_ticker = candidate
                break
        except Exception:
            continue

    cleaned = []
    for item in raw_news[:limit]:
        content = item.get("content", {}) if isinstance(item, dict) else {}
        provider = content.get("provider", {}) if isinstance(content, dict) else {}
        canonical_url = content.get("canonicalUrl", {}) if isinstance(content, dict) else {}
        click_url = content.get("clickThroughUrl", {}) if isinstance(content, dict) else {}

        title = content.get("title") or item.get("title", "Untitled")
        source = provider.get("displayName") or content.get("provider") or item.get("publisher") or "Unknown"
        url = canonical_url.get("url") or click_url.get("url") or item.get("link", "")

        published_at = "N/A"
        pub_date = content.get("pubDate")
        if pub_date:
            try:
                published_at = datetime.fromisoformat(pub_date.replace("Z", "+00:00")).strftime("%Y-%m-%d %H:%M")
            except Exception:
                pass
        elif item.get("providerPublishTime"):
            try:
                published_at = datetime.fromtimestamp(int(item["providerPublishTime"])).strftime("%Y-%m-%d %H:%M")
            except Exception:
                pass

        cleaned.append(
            {
                "title": title,
                "source": source,
                "url": url,
                "published_at": published_at,
            }
        )

    return final_ticker, cleaned


ticker, news_items = fetch_stock_news(stock_input, top_n)
print(f"Ticker used: {ticker} | Headlines fetched: {len(news_items)}")

Ticker used: RELIANCE.NS | Headlines fetched: 10


In [4]:
news_df = pd.DataFrame(news_items)
news_df[["title", "source", "published_at", "url"]].head(top_n) if not news_df.empty else news_df

,title,source,published_at,url
0,OpenAI Appoints JioStar CEO to Lead Asia Expan...,GuruFocus.com,2026-03-26 12:28,https://finance.yahoo.com/sectors/technology/a...
1,Exclusive-India's Reliance buys 5 million barr...,Reuters,2026-03-24 10:06,https://sg.finance.yahoo.com/news/exclusive-in...
2,Factbox-Ambani's Reliance Jio: businesses and ...,Reuters,2026-03-23 09:38,https://sg.finance.yahoo.com/news/factbox-amba...
3,India Coca‑Cola bottler SLMG says Middle East ...,Reuters,2026-03-23 04:53,https://sg.finance.yahoo.com/news/india-coca-c...
4,How The Reliance Industries (NSEI:RELIANCE) St...,Simply Wall St.,2026-03-20 10:06,https://finance.yahoo.com/markets/stocks/artic...
5,"Factbox-India, a major energy consumer and ref...",Reuters,2026-03-19 08:46,https://sg.finance.yahoo.com/news/factbox-indi...
6,"India orders oil, gas firms to share import, e...",Reuters,2026-03-18 19:52,https://sg.finance.yahoo.com/news/india-orders...
7,"Reliance Industries, Samsung subsidiary sign $...",ESG Dive,2026-03-18 10:37,https://www.esgdive.com/news/reliance-industri...
8,Robert Kiyosaki Says India's Reliance Could Be...,Benzinga,2026-03-13 22:30,https://finance.yahoo.com/news/robert-kiyosaki...
9,Trump says the U.S. will open its first new oi...,Fortune,2026-03-11 19:15,https://finance.yahoo.com/news/trump-says-u-op...
